<a href="https://colab.research.google.com/github/suleman-khawaja/diabetes-prediction-deep-learning/blob/main/Diabetes%20Prediction%20System%20using%20Deep%20Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

## ⚠️ Medical Disclaimer

> **IMPORTANT NOTICE:**  
> This project is created strictly for **educational, academic, and demonstration purposes only**. It is **NOT** intended to be a diagnostic tool, medical advice, or a replacement for professional healthcare evaluation.
>
> 1. **No Medical Endorsement:** The predictions generated by this machine learning model should not be used to diagnose, treat, or manage diabetes or any other health condition.
> 2. **Dataset Limitations:** The model was trained on a small, specific historical dataset (PIMA Indians Diabetes Dataset) and may not generalize accurately to broader real-world clinical populations.
> 3. **Use with Caution:** Always consult a qualified healthcare professional or clinical diagnostic laboratory for medical concerns and testing.
>
> The author assumes **no liability or responsibility** for any clinical decisions or outcome predictions made using this code or model.

In [4]:
# ------------------------------------------------------------------------------
# PROJECT IMPORTS
# ------------------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Scikit-Learn (Data Splitting, Scaling & Evaluation)
import sklearn as sk
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# TensorFlow / Keras (Deep Learning Models & Layers)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.regularizers import l2

In [5]:
file="/content/diabetesset.xlsx"

df = pd.read_excel(file)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                                                                    Non-Null Count  Dtype  
---  ------                                                                    --------------  -----  
 0   Number of times pregnant                                                  768 non-null    int64  
 1   Plasma glucose concentration a 2 hours in an oral glucose tolerance test  768 non-null    int64  
 2   Diastolic blood pressure (mm Hg)                                          768 non-null    int64  
 3   Triceps skin fold thickness (mm)                                          768 non-null    int64  
 4   2-Hour serum insulin (mu U/ml)                                            768 non-null    int64  
 5   Body mass index (weight in kg/(height in m)^2)                            768 non-null    float64
 6   Diabetes pedigree function                                         

### Descriptive Statistics

Let's get a statistical summary of the dataset to understand the distribution of each feature. This can help identify potential issues like outliers or incorrect data entries (e.g., 0 values in columns like 'Glucose' or 'BloodPressure').

In [6]:
display(df.describe())

,Number of times pregnant,Plasma glucose concentration a 2 hours in an oral glucose tolerance test,Diastolic blood pressure (mm Hg),Triceps skin fold thickness (mm),2-Hour serum insulin (mu U/ml),Body mass index (weight in kg/(height in m)^2),Diabetes pedigree function,Age (years),Class variable
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [7]:
print(df.isnull().sum())

Number of times pregnant                                                    0
Plasma glucose concentration a 2 hours in an oral glucose tolerance test    0
Diastolic blood pressure (mm Hg)                                            0
Triceps skin fold thickness (mm)                                            0
2-Hour serum insulin (mu U/ml)                                              0
Body mass index (weight in kg/(height in m)^2)                              0
Diabetes pedigree function                                                  0
Age (years)                                                                 0
Class variable                                                              0
dtype: int64


In [8]:
(df==0).sum()

,0
Number of times pregnant,111
Plasma glucose concentration a 2 hours in an oral glucose tolerance test,5
Diastolic blood pressure (mm Hg),35
Triceps skin fold thickness (mm),227
2-Hour serum insulin (mu U/ml),374
Body mass index (weight in kg/(height in m)^2),11
Diabetes pedigree function,0
Age (years),0
Class variable,500


In [9]:
zeros_column=['Plasma glucose concentration a 2 hours in an oral glucose tolerance test','Diastolic blood pressure (mm Hg)', 'Triceps skin fold thickness (mm)', '2-Hour serum insulin (mu U/ml)', 'Body mass index (weight in kg/(height in m)^2)' ]
df[zeros_column]=df[zeros_column].replace(0, np.nan)
df[zeros_column]=df[zeros_column].fillna(df[zeros_column].median())

In [10]:
df.isnull().sum()

,0
Number of times pregnant,0
Plasma glucose concentration a 2 hours in an oral glucose tolerance test,0
Diastolic blood pressure (mm Hg),0
Triceps skin fold thickness (mm),0
2-Hour serum insulin (mu U/ml),0
Body mass index (weight in kg/(height in m)^2),0
Diabetes pedigree function,0
Age (years),0
Class variable,0


In [11]:
x=df.drop('Class variable', axis=1)
y=df['Class variable']

In [12]:
print(x.shape)
print(y.shape)

(768, 8)
(768,)


In [13]:
x_train,x_,y_train,y_=train_test_split(x,y, random_state=1, test_size=0.3)
x_cv,x_test,y_cv,y_test=train_test_split(x_,y_,random_state=1, test_size=0.5)
print(x_train.shape, y_train.shape)
print(x_cv.shape, y_cv.shape)
print(x_test.shape, y_test.shape)


(537, 8) (537,)
(115, 8) (115,)
(116, 8) (116,)


In [14]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_cv_scaled=scaler.transform(x_cv)
x_test_scaled=scaler.transform(x_test)

In [15]:
model=tf.keras.models.Sequential(
    [
        Dense(units=12, activation='relu', name='L1', kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        Dense(units=6, activation='relu', name='L2', kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        Dense(units=1, activation='linear', name='L3')
    ]
)
model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    metrics=['accuracy']
)

In [16]:
model.fit(
    x_train_scaled,y_train,
    epochs=200,
    validation_data=(x_cv_scaled,y_cv)
)

Epoch 1/200
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.6592 - loss: 0.7829 - val_accuracy: 0.6522 - val_loss: 0.6349
Epoch 2/200
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6592 - loss: 0.6311 - val_accuracy: 0.6522 - val_loss: 0.5639
Epoch 3/200
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6592 - loss: 0.5931 - val_accuracy: 0.6522 - val_loss: 0.5321
Epoch 4/200
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6592 - loss: 0.5633 - val_accuracy: 0.6522 - val_loss: 0.5154
Epoch 5/200
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6592 - loss: 0.5453 - val_accuracy: 0.6522 - val_loss: 0.4982
Epoch 6/200
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6592 - loss: 0.5344 - val_accuracy: 0.6522 - val_loss: 0.4916
Epoch 7/200
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6592 - loss: 0.5240 - val_accuracy: 0.6522 - val_loss: 0.4861
Epoch 8/200
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6592 - loss: 0.5156 - val_accuracy: 0.6522 - 

In [17]:
test_loss, test_acc= model.evaluate(x_test_scaled,y_test)
print(test_loss, test_acc)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8103 - loss: 0.4190 
0.41904160380363464 0.8103448152542114


Now we are making some predictions on real data made randomly

In [18]:
#we are placing numbers and data according to columns you can see in df.info() code
feature_cols = df.drop(columns=['Class variable']).columns

new_patient = pd.DataFrame([[2, 130, 70, 20, 80, 25.5, 0.35, 28]], columns=feature_cols)
new_patient_scaled = scaler.transform(new_patient)

In [19]:
logits=model.predict(new_patient_scaled)
probability=tf.nn.sigmoid(logits).numpy()



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step


In [20]:
print(f"The Probability in % is : {probability * 100}")
yhat=(probability >= 0.5).astype(int)
print(f"Output Means 1=Diabetic, 0=No Diabetes, {yhat}")

The Probability in % is : [[11.905807]]
Output Means 1=Diabetic, 0=No Diabetes, [[0]]


In [21]:

test_logits = model.predict(x_test_scaled)
test_preds = (tf.nn.sigmoid(test_logits).numpy() >= 0.5).astype(int)

# Confusion Matrix and Classification Report displayed
print("Confusion Matrix:")
print(confusion_matrix(y_test, test_preds))

print("\nClassification Report:")
print(classification_report(y_test, test_preds))

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
Confusion Matrix:
[[62  9]
 [11 34]]

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.87      0.86        71
           1       0.79      0.76      0.77        45

    accuracy                           0.83       116
   macro avg       0.82      0.81      0.82       116
weighted avg       0.83      0.83      0.83       116

